## Proyecto Final: Sistema de Recomendación E-commerce
### Modelo 1: Filtrado colaborativo

**Equipo:** MetricEdge

**Autor:** Sara Henao

**Dataset:** interacciones_clean.csv

Fuente de origen: E-commerce Sales & Customer Analytics (150k), Kaggle

### Objetivo de este notebook
Entrenamiento del primer modelo por filtrado colaborativo, basado en el principio "los usuarios que tuvieron comportamientos parecidos en el pasado, probablemente compartirán preferencias en el futuro"

**Importante:** A diferencia de los métodos basados en contenido (que se apoyan en las características de los productos o en el perfil del usuario), el filtrado colaborativo aprende directamente del comportamiento colectivo: compras, calificaciones, clics o reproducciones. 

In [2]:
#Importar librerías
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import train_test_split, precision_at_k, ranking_metrics_at_k
import scipy.sparse as sparse

import warnings
warnings.filterwarnings('ignore')

In [13]:
#Cargar datos interacciones_clean.csv

df_interactions = pd.read_csv('../data/processed/interacciones_clean.csv')

display(df_interactions.head())


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,customer_id,order_date,order_status
0,ORD-100004,PROD-000475,1,12.91,0.094500,1.22,12.91,2.10,10.10,23.89,4.26,9.53,CUST-022489,2025-03-29,Completed
1,ORD-100004,PROD-000782,1,116.11,0.098958,11.49,116.11,18.83,2.30,125.75,67.97,55.48,CUST-022489,2025-03-29,Completed
2,ORD-100004,PROD-000912,1,335.86,0.042012,14.11,335.86,57.92,4.06,383.73,195.75,183.92,CUST-022489,2025-03-29,Completed
3,ORD-100016,PROD-000319,2,176.23,0.180701,63.69,352.46,20.21,3.80,312.78,157.12,151.86,CUST-011235,2023-03-07,Completed
4,ORD-100016,PROD-001037,1,61.00,0.090164,5.50,61.00,3.88,15.15,74.53,24.60,34.78,CUST-011235,2023-03-07,Completed


**Concepto de entrenamiento:**

**Datos de Entrada:** Solo ID de Usuario, ID de Producto e Interacción (cantidad de cada producto comprado o rating de producto)

**¿Qué aprende?** Similitudes o Factores latentes (gustos abstractos)

**Mecanismo de "Entrenamiento":** Factorización de matrices SVD(Singular Value Decomposition)/ALS(Alternating Least Squares) o cálculo de distancias (Coseno) KNN

Para este modelo se escoge factorización de matrices, ya que tiene la ventaja de ser escalable a millones de usuarios y productos sin depender de información textual, permite descubrir relaciones complejas que el método del coseno ignora y tiene un mejor manejo de de la dispersión (sparsity) al capturar las tendencias generales, incluyendo a los articulos de venta y de nicho (long tail). Además, se selecciona el algoritmo ALS optimizado para datos implícitos. La elección se justifica debido a que el dataset en la tabla *interactions* cuenta con más de 755000 transacciones, donde la interacción clave es el comportamiento/compra (quantity) y no de calificaciones individuales de productos(rating directos por el usuario). Este último dato que no se tiene actualmente, lo que se tiene es un rating global por producto, lo que no permite rastrear la satisfacción del usuario individual y por ende limitando la aplicación de un algoritmo como SVD, que normalmente se usa cuando el objetivo del negocio es predecir la satisfacción exacta (calificación) del cliente.

Otro factor a tener en cuenta es el comportamiento matemático de los algoritmos. Al trabajar con una matriz pivotada, se generarán vacíos, aquí es donde difieren las interpretaciones de ALS y SVD. ALS toma esos vacíos generados y los transforma en "0", interpretandolos como un desinterés potencial o falta de exposición del artículo, en cambio SVD
toma esos NaN, los ignora y solo entrena con las celdas que sí tienen datos, ya que fue diseñado para datos explícitos (estrellas, reviews, ratings,etc)


El valor del filtrado colaborativo radica en su capacidad de aprender gustos implícitos sin conocer las características del producto, lo que lo hace aplicable en contextos donde los metadatos son incompletos o heterogéneos

In [14]:
# Filtrar estados de orden válidos si aplica (ej. solo 'Completed')
if 'order_status' in df_interactions.columns:
    df_interactions = df_interactions[df_interactions['order_status'] == 'Completed']
    
#Dataframe con las columnas necesarias para el modelo 'customer_id','product_id','quantity'
df_model = df_interactions[['customer_id','product_id','quantity']].copy()

#consolidar si un cliente  compró el mismo producto en diferentes órdenes
df_model = df_model.groupby(['customer_id','product_id'])['quantity'].sum().reset_index()

print(f"Registros únicos tras la consolidación: {df_model.shape[0]}")

Registros únicos tras la consolidación: 325133


**Matriz dispersa**

La librería implicit no entiende los IDs de clientes o productos si estos son cadenas de texto (CUST_10293 o PROD_ABC). Necesita que se conviertan a números enteros indexados (0, 1, 2, 3...). Además, al hacer una tabla pivote tradicional con .pivot(), Python intentaría guardar millones de ceros en la memoria RAM. Para evitarlo, se utiliza una Matriz Dispersa en formato CSR (Compressed Sparse Row), la cual solo almacena en memoria las celdas que sí tienen datos

In [5]:
# 1. Convertir IDs de texto a categorías numéricas (Factores/Categorías)
df_model['customer_id'] = df_model['customer_id'].astype("category")
df_model['product_id'] = df_model['product_id'].astype("category")

# 2. Crear columnas con los códigos numéricos indexados
df_model['user_code'] = df_model['customer_id'].cat.codes
df_model['item_code'] = df_model['product_id'].cat.codes

# 3. Guardar diccionarios de mapeo para poder recuperar los IDs reales una vez el modelo esté entrenado y recomiende
# (Mapeo de código numérico -> ID original de Kaggle)
user_map = dict(enumerate(df_model['customer_id'].cat.categories))
item_map = dict(enumerate(df_model['product_id'].cat.categories))

# 4. Crear la Matriz Dispersa en formato CSR (Compressed Sparse Row)
# Sintaxis de SciPy: csr_matrix((datos, (filas, columnas)))
user_item_matrix = sparse.csr_matrix(
    (df_model['quantity'].astype(float), (df_model['user_code'], df_model['item_code']))
)

print(f"Dimensiones de la matriz (Usuarios x Productos): {user_item_matrix.shape}")
print(f"Porcentaje de densidad (celdas llenas): {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100:.4f}%")


Dimensiones de la matriz (Usuarios x Productos): (24748, 1175)
Porcentaje de densidad (celdas llenas): 1.1181%


**Entrenamiento  y prueba del modelo**

Para evaluar el modelo de filtrado colaorativo ALS, no se puede realizar un train_test_split como se realiza en otros modelos, se debe hacer una división basada en ocultar interacciones o máscara/masking

Ajustes de calibración del modelo
- La penalización de la "Falta de Ceros" en Quantity: Por defecto, el algoritmo ALS asume que si el valor de compra es 1, la confianza es baja, y si es 10, es alta. En el dataset la gran mayoría de los clientes compraron solo 1 unidad de un producto, como se observó en el EDA primario, para ALS esa señal es casi tan débil como un cero.

Por este motivo se debe escalar la matriz con un factor alfa (α) para que el modelo reintreprete que "si hay una compra, aumenta drásticamente mi confianza de que le gusta". El estándar es multiplicar la matriz de entrenamiento por un valor entre 15 y 40.

In [6]:
#Prueba 1: Matriz original 
#El 20% de las interacciones se ocultan para hacer la validación posterior
train_matrix, test_matrix = train_test_split(user_item_matrix, train_percentage=0.8, random_state=42)

# Multiplicamos la matriz de entrenamiento por un factor alfa (ej. 40)
# Esto le indica al algoritmo que una compra de cantidad 1 SÍ es una señal fuerte.
alpha_val = 40
train_matrix_scaled = train_matrix.multiply(alpha_val).astype('float32').tocsr() #(float32)Requerimiento de implicit para que funcione


#Configurar el modelo ALS
model_ALS = AlternatingLeastSquares(
    factors=128,          # Número de factores latentes (vectores ocultos) estándar entre 64 y 128. Muy bajo|underfitting
                            #muy alto| overffiting 
    regularization=5.0,  # Penalización para evitar sobreajuste (overfitting). Estandar 0.01, 0.1, 1.0. 
                            #si falla drásticamente al recomendar cosas nuevas, debes subir este valor 
    iterations=30,       # Cuántas iteraciones hará para ajustar las matrices de usuarios y productos
                            #entre 15 y 30 iteraciones son suficientes para que el modelo converja
    random_state=42      # Semilla para que los resultados sean reproducibles
)

# Entrenar el modelo 
model_ALS.fit(train_matrix_scaled)

#Evaluar utilizando las matrices en formato natural (Usuarios x Ítems)
# Precision@K mide: De los top 10 productos que le recomendamos al usuario, ¿cuántos compró realmente en el test_matrix?
#Forzar formato CSR explícito en la evaluación
metricas_nativas = ranking_metrics_at_k(
    model=model_ALS, 
    train_user_items=train_matrix.tocsr(), # Matriz Train base (Usuarios x Ítems) Formato consistente con el .fit()
    test_user_items=test_matrix.tocsr(),   # Matriz Test base (Usuarios x Ítems)Formato consistente con el .fit()
    K=10,
    show_progress=True
)

# 7. EXTRACCIÓN DE RESULTADOS FINALES CORPORATIVOS
p_at_k = metricas_nativas.get('precision', 0)
map_at_k = metricas_nativas.get('map', 0)
ndcg_at_k = metricas_nativas.get('ndcg', 0)
auc_at_k = metricas_nativas.get('auc', 0)

print(f"\n--- REPORTE DE RENDIMIENTO (METRICEDGE) ---")
print(f"La Precisión@10 del modelo es: {p_at_k * 100:.2f}%")
print(f"El MAP@10 del modelo es:        {map_at_k * 100:.2f}%")
print(f"El NDCG@10 del modelo es:       {ndcg_at_k * 100:.2f}%")
print(f"El AUC del modelo es:           {auc_at_k * 100:.2f}%")

100%|██████████| 21755/21755 [00:04<00:00, 4536.88it/s]


--- REPORTE DE RENDIMIENTO (METRICEDGE) ---
La Precisión@10 del modelo es: 0.86%
El MAP@10 del modelo es:        0.26%
El NDCG@10 del modelo es:       0.54%
El AUC del modelo es:           50.01%


In [15]:
#Prueba 2: matriz binaria

# 1. División Train / Test en formato natural (Usuarios x Ítems)
train_matrix, test_matrix = train_test_split(user_item_matrix, train_percentage=0.8, random_state=42)

# 2. Señal Binaria Pura: Creamos la matriz de presencia/ausencia para el Fit
train_matrix_binary = train_matrix.copy()
train_matrix_binary.data = np.ones_like(train_matrix_binary.data)
train_matrix_binary = train_matrix_binary.astype('float32').tocsr() # Formato exigido por implicit

# 3. CONFIGURACIÓN COMPLETA DEL MODELO ALS CON HIPERPARÁMETROS NATIVOS
model_ALS = AlternatingLeastSquares(
    factors=64,          
    regularization=0.01,         # <-- AJUSTADO: Menos penalización para permitir mayor flexibilidad
    alpha=40.0,                  # <-- AJUSTADO: Pasamos el peso de confianza de forma nativa al motor
    iterations=25,               # Subimos ligeramente las iteraciones para dar tiempo de convergencia
    calculate_training_loss=True, # Activamos la auditoría de pérdida en consola
    random_state=42      
)

# 4. ENTRENAMIENTO DIRECTO (Formato natural Usuarios x Ítems sin .T)
# Alimentamos la matriz binaria limpia; el parámetro 'alpha=40.0' hará el escalado internamente
model_ALS.fit(train_matrix_binary)

# 5. EVALUACIÓN NATIVA DIRECTA (Formato natural Usuarios x Ítems sin .T)
metricas_nativas = ranking_metrics_at_k(
    model=model_ALS, 
    train_user_items=train_matrix.tocsr(), # Matriz Train base en su formato original
    test_user_items=test_matrix.tocsr(),   # Matriz Test base en su formato original
    K=10,
    show_progress=True
)

# 6. EXTRACCIÓN DE RESULTADOS FINALES CORPORATIVOS
p_at_k = metricas_nativas.get('precision', 0)
map_at_k = metricas_nativas.get('map', 0)
ndcg_at_k = metricas_nativas.get('ndcg', 0)
auc_at_k = metricas_nativas.get('auc', 0)

print(f"\n--- REPORTE DE RENDIMIENTO CON HYPERPARAMETER TUNING (METRICEDGE) ---")
print(f"La Precisión@10 del modelo es: {p_at_k * 100:.2f}%")
print(f"El MAP@10 del modelo es:        {map_at_k * 100:.2f}%")
print(f"El NDCG@10 del modelo es:       {ndcg_at_k * 100:.2f}%")
print(f"El AUC del modelo es:           {auc_at_k * 100:.2f}%")


100%|██████████| 21755/21755 [00:02<00:00, 9321.66it/s] 


--- REPORTE DE RENDIMIENTO CON HYPERPARAMETER TUNING (METRICEDGE) ---
La Precisión@10 del modelo es: 0.87%
El MAP@10 del modelo es:        0.27%
El NDCG@10 del modelo es:       0.56%
El AUC del modelo es:           50.00%


In [8]:
#Bloque de auditoría
# 1. Contar cuántos productos únicos tiene cada cliente
productos_por_cliente = df_model.groupby('customer_id')['product_id'].count()

# 2. Calcular cuántos clientes tienen exactamente 1 solo producto comprado
clientes_con_un_solo_producto = (productos_por_cliente == 1).sum()
total_clientes = len(productos_por_cliente)
porcentaje_un_producto = (clientes_con_un_solo_producto / total_clientes) * 100

print(f"Total de clientes en el dataset: {total_clientes}")
print(f"Clientes que solo compraron 1 único producto: {clientes_con_un_solo_producto}")
print(f"Porcentaje de clientes con solo 1 producto: {porcentaje_un_producto:.2f}%\n")

# 3. Ver una distribución rápida (Percentiles)
print("Distribución del número de productos comprados por cliente:")
print(productos_por_cliente.describe(percentiles=[0.25, 0.5, 0.75, 0.90]))

Total de clientes en el dataset: 24748
Clientes que solo compraron 1 único producto: 205
Porcentaje de clientes con solo 1 producto: 0.83%

Distribución del número de productos comprados por cliente:
count    24748.000000
mean        13.137749
std          6.751342
min          1.000000
25%          8.000000
50%         12.000000
75%         17.000000
90%         22.000000
max         51.000000
Name: product_id, dtype: float64


Conclusión:

Tras un riguroso proceso de implementación, optimización y calibración de hiperparámetros del algoritmo Alternating Least Squares (ALS), el equipo MetricEdge concluye que el filtrado colaborativo puro es estadísticamente ineficaz para modelar este conjunto de datos debido a la dispersión estructural intrínseca del historial minorista (Sparsity Cold-Start masivo). El estancamiento del modelo en un área bajo la curva del AUC del ~50% y una Precisión@10 menor al 1%, se debe a:

- El análisis exploratorio de datos (EDA) reveló que el 50% de la masa crítica de clientes (24,838 usuarios) ha adquirido 13 productos o menos en toda su historia transaccional. Esta falta de profundidad histórica impide que el algoritmo construya vectores estables para cada individuo.

- Al aplicar la separación del set de entrenamiento y prueba mediante train_test_split (80/20), el modelo se reduce a entrenar con apenas 9 o 10 interacciones físicas por usuario. Con ese mínimo historial, el optimizador queda forzado a "adivinar" 1 o 2 ítems específicos que fueron ocultados dentro de un mar de 1,175 productos disponibles en el catálogo, reduciendo la probabilidad matemática de acierto al azar estadístico.

- Con una densidad matricial de apenas el 1.20%, el filtrado colaborativo puro sufre de "ceguera contextual". Al no apoyarse en metadatos (como marcas, categorías o precios) y depender exclusivamente de la co-ocurrencia de comportamiento, las filas de la matriz no logran conectarse entre sí. El espacio latente colapsa y el recomendador genera vectores planos e idénticos para la mayoría de los usuarios, devolviendo un carrusel homogéneo basado únicamente en los artículos más populares de la tienda, lo que equivale a adivinar al azar (50.06% AUC).



Este resultado fundamenta científicamente la necesidad de descartar las arquitecturas basadas exclusivamente en comportamiento colectivo para este caso de uso. Como estrategia de negocio para MetricEdge, esta conclusión justifica técnicamente la transición hacia el desarrollo de Modelos Basados en Contenido (Content-Based) o Sistemas Híbridos. Estas soluciones permitirán romper el problema de la dispersión al indexar los metadatos de Kaggle (categorías y atributos de producto), garantizando recomendaciones personalizadas efectivas desde la primera interacción sin depender del solapamiento de compras entre clientes.

Nota: Quitaron el Recall porque en sistemas de recomendación con catálogos masivos y datos implícitos, el Recall no mide el valor comercial real.
El Recall te premia únicamente si logras descubrir todos los productos que el usuario compró en el set de prueba. Si un usuario compró 15 artículos distintos, para tener un Recall del 100% estás obligado a recomendarle exactamente esos 15 artículos.El castigo al tamaño del Top-K: Como tú estás midiendo un Top-10 (K=10), es matemáticamente imposible que un usuario con más de 10 compras en el set de prueba alcance un buen Recall, penalizando la métrica injustamente.

La alternativa moderna (NDCG y MAP): En el e-commerce real, lo que le importa al negocio es que, si le muestras 10 productos al cliente en la pantalla, los mejores y más relevantes aparezcan en los primeros lugares (posiciones 1 a 3). Por eso la librería calcula de forma fija MAP y NDCG (Métricas de ganancia acumulada descontada), las cuales ignoran el Recall y se enfocan en penalizar si pusiste un producto relevante al final de la lista en lugar del principio.